# Persiapan API  
Pada tahap inisiasi, kita perlu mengimpor pustaka openai untuk membuka jalur komunikasi dengan model. Penggunaan pustaka dotenv sangat krusial di sini. Menyimpan kunci API (API Key) secara langsung (hardcode) di dalam naskah kode merupakan celah keamanan yang sangat fatal, terutama jika kode tersebut nantinya diunggah ke repositori publik. Praktik terbaik yang kita terapkan adalah menyimpan kunci tersebut pada berkas tersembunyi bernama .env dan memuatnya ke dalam lingkungan kerja secara aman.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Memuat konfigurasi variabel lingkungan secara aman dari berkas .env
load_dotenv(override=True)

# Mengambil kunci akses API OpenAI ke dalam variabel
openai_api_key = os.getenv('OPENAI_API_KEY')

# Validasi sederhana untuk memastikan kunci telah terbaca oleh sistem
if openai_api_key:
    print("Sistem mendeteksi kunci API OpenAI yang valid.")
else:
    print("Peringatan: Kunci API OpenAI belum terkonfigurasi pada sistem.")

Sistem mendeteksi kunci API OpenAI yang valid.


# Konfigurasi API GPT dan Lokal (Ollama)
Ini merupakan teknik rekayasa perangkat lunak yang sangat efisien dan elegan. Pustaka openai di Python ternyata dibangun dengan arsitektur yang sangat modular. Kita dapat menggunakannya untuk memanggil layanan OpenAI resmi, atau secara cerdas mengubah parameter base_url menuju localhost (komputer kita sendiri) untuk mengakses model Ollama secara luring (offline). Dengan arsitektur Ollama yang meniru titik akhir (endpoint) REST milik OpenAI, kita hanya perlu mempertahankan satu standar penulisan kode untuk dua sumber model yang sama sekali berbeda.

In [2]:
# 1. Inisialisasi klien akses untuk layanan komersial OpenAI
openai = OpenAI(api_key=openai_api_key)

# 2. Inisialisasi klien akses untuk layanan lokal Ollama
# Parameter base_url diarahkan ke port default Ollama di sistem lokal
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# Pengujian Inferensi Dasar (Inference)
Untuk menginstruksikan model, struktur data yang kita kirimkan tidak boleh berupa teks biasa, melainkan harus diformat sebagai senarai kamus (list of dictionaries). Di dalamnya memuat entitas pengirim (role, seperti "user" atau "system") dan substansi instruksi (content). Pada sel ini, kita melakukan uji coba inferensi untuk meminta model menyelesaikan teka-teki logika probabilistik. Kita juga menggunakan parameter reasoning_effort (khusus untuk model penalaran OpenAI terbaru) untuk mengefisienkan atau membatasi durasi komputasi model saat berpikir.

In [4]:
# Mendefinisikan struktur prompt untuk evaluasi logika
easy_puzzle = [
    {"role": "user", "content": "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."}
]

# Melakukan panggilan API menuju model GPT dengan batas penalaran minimal
response_gpt = openai.chat.completions.create(
    model="gpt-4.1-mini", 
    messages=easy_puzzle, 
)

print("Keluaran dari model OpenAI:", response_gpt.choices[0].message.content)

Keluaran dari model OpenAI: 2/3


# Peralihan menuju Inferensi Lokal 
Sebagai langkah mitigasi apabila alokasi dana API kita menipis, atau ketika kita sedang melakukan tahap prototyping yang intens, kita dapat segera mengeksekusi inferensi pada Ollama. Prasyarat pelaksanaannya cukup mudah: kita hanya perlu mengunduh parameter model (misalnya, llama3.2) ke dalam perangkat keras kita melalui antarmuka baris perintah (Terminal/CMD) dengan perintah ollama pull llama3.2. Setelah itu, eksekusi kode dilakukan melalui instrumen klien Ollama yang telah kita siapkan di Sel 3.

In [5]:
# Meminta komputasi model lokal tanpa memerlukan koneksi internet maupun biaya API
response_lokal = ollama.chat.completions.create(
    model="llama3.2", 
    messages=easy_puzzle
)

print("Keluaran dari model lokal (Ollama):", response_lokal.choices[0].message.content)

Keluaran dari model lokal (Ollama): 1/2


# Abstraksi dan Pengawasan Anggaran dengan LiteLLM
Dalam skala produksi, pengawasan biaya (cost tracking) sangatlah krusial. Oleh sebab itu, integrasi pustaka LiteLLM amat berguna. Perangkat lunak perantara ini mendobrak kerumitan dengan menstandarisasi pemanggilan menuju lebih dari seratus penyedia model berbeda. Lebih dari itu, ia secara otomatis menangkap, menyajikan kalkulasi metrik penggunaan token, dan memberikan estimasi nilai finansial secara presisi pada setiap eksekusi.

In [10]:
from litellm import completion

# Menyiapkan instruksi sederhana
tell_a_joke = [{"role": "user", "content": "Ceritakan sebuah lelucon ringan untuk seorang insinyur perangkat lunak. Jangan menyinggung siapapun, dan pastikan leluconnya mudah dipahami."}]

# Melakukan inferensi model melalui lapisan abstraksi LiteLLM
response_lite = completion(model="openai/gpt-4.1-mini", messages=tell_a_joke)

print("Respons teks:", response_lite.choices[0].message.content)

# Peninjauan otomatis terhadap utilitas token dan beban biaya secara langsung
print(f"Total pemakaian token: {response_lite.usage.total_tokens}")
print(f"Beban finansial tercatat: {response_lite._hidden_params['response_cost']:.4f} USD")

Respons teks: Tentu! Berikut lelucon ringan untuk seorang insinyur perangkat lunak:

Kenapa programmer selalu bingung saat menghadapi jalan buntu?  
Karena mereka terlalu terbiasa dengan *break* dan *continue*!
Total pemakaian token: 87
Beban finansial tercatat: 0.0001 USD


# Pengenalan Tingkat Lanjut Terhadap LangChain
LangChain adalah kerangka kerja tingkat tinggi (high-level framework) yang sangat esensial saat kita hendak membangun aplikasi AI yang berarsitektur kompleks. Alih-alih mengelola panggilan API secara manual, LangChain menyajikan abstraksi fungsional. Melalui modul langchain_openai, kita dapat memfasilitasi komunikasi menuju model bahasa dengan sintaks berorientasi objek. Penggunaan metode invoke di sini merupakan pijakan awal yang nantinya sangat bermanfaat untuk merangkai AI dengan memori eksternal, dokumen PDF, atau pangkalan data (database).

In [21]:
# Mengimpor modul spesifik OpenAI dari kerangka kerja LangChain
from langchain_openai import ChatOpenAI

# Menginisialisasi objek representasi model bahasa (LLM)
llm = ChatOpenAI(model="gpt-4.1-mini")

# Mengeksekusi instruksi melalui metode abstraksi invoke khas LangChain
response_langchain = llm.invoke(tell_a_joke)

print("Keluaran via kerangka kerja LangChain:\n", response_langchain.content)

Keluaran via kerangka kerja LangChain:
 Tentu! Berikut lelucon ringan untuk insinyur perangkat lunak:

Kenapa programmer selalu bingung saat di hutan?  
Karena mereka takut *loop* yang nggak ada ujungnya! 😄


# Simulasi Dua AI yang Saling Berinteraksi (Menjaga Konteks Percakapan)

Pada bagian ini, kita ingin memahami bagaimana AI bisa terlihat “ingat” percakapan sebelumnya.

Sebenarnya, model AI seperti GPT tidak memiliki memori bawaan. Setiap kali kita mengirim pesan ke API, model menganggap itu sebagai percakapan baru. Agar AI tetap memahami konteks, kita harus mengirim ulang seluruh riwayat percakapan sebelumnya setiap kali ada pesan baru. Jadi, sistemlah yang menyimpan dan menambahkan (append) percakapan lama ke dalam daftar pesan sebelum dikirim kembali ke model.

Dalam percobaan ini, kita membuat dua peran berbeda:

Model GPT diatur menjadi agen yang suka berdebat dan menyampaikan argumen.

Model lokal Ollama diatur menjadi agen yang lebih santun dan berperan sebagai penengah.

Setiap kali salah satu AI menjawab, jawabannya ditambahkan ke riwayat percakapan, lalu dikirim kembali ke AI lainnya. Dengan cara ini, percakapan bisa terus berlanjut dan tetap terasa konsisten, meskipun sebenarnya model tidak benar-benar memiliki memori sendiri.

In [ ]:
# 1. Mendefinisikan profil karakter agen dengan topik filosofi pengembangan perangkat lunak
profil_gpt = """Anda adalah pengembang perangkat lunak senior yang pragmatis, argumentatif, dan sering berbicara dengan nada sinis. 
Dalam perdebatan ini, Anda berpendapat bahwa menulis kode dengan cepat untuk segera menyelesaikan dan merilis fitur (meskipun strukturnya berantakan) adalah hal yang paling krusial demi mengejar tenggat waktu bisnis. 
Anda sangat kritis terhadap programmer yang menghabiskan terlalu banyak waktu untuk merapikan kode (over-engineering), karena menurut Anda hal tersebut lamban dan tidak menghasilkan nilai tambah langsung bagi pengguna."""

profil_ollama = """Anda adalah arsitek perangkat lunak yang santun, sabar, dan sangat menjunjung tinggi kualitas. 
Dalam perdebatan ini, Anda berpendapat bahwa menulis kode dengan struktur yang rapi, arsitektur yang bersih (clean code), dan dokumentasi yang baik adalah hal yang mutlak, meskipun memakan waktu penyelesaian yang jauh lebih lama. 
Anda selalu menanggapi kritik dengan logis dan ramah, menekankan bahaya fatal dari utang teknis (technical debt) serta pentingnya kemudahan pemeliharaan sistem (maintainability) oleh tim di masa depan."""

# 2. Inisialisasi basis data interaksi berurutan (State Management) sebagai pemantik awal diskusi
rekaman_interaksi_gpt = ["Di dunia nyata, pengguna tidak peduli seberapa rapi struktur kode kita di belakang layar. Yang penting fitur selesai dengan cepat, aplikasi berjalan, dan tenggat waktu terpenuhi. Terlalu pusing memikirkan struktur hanya akan membuat kita tertinggal dari kompetitor!"]
rekaman_interaksi_ollama = ["Memang benar kecepatan rilis itu penting untuk bisnis. Namun, jika kita menulis kode asal cepat dan berantakan, kita sedang menabung masalah besar. Beberapa bulan ke depan, saat ada fitur baru atau celah kerusakan (bug), kode yang berantakan tersebut akan sangat sulit dan memakan waktu lama untuk diperbaiki."]

# 3. Mendefinisikan fungsi transmisi untuk Agen GPT
def invoke_gpt_agent():
    beban_pesan = [{"role": "system", "content": profil_gpt}]
    # Sinkronisasi konteks percakapan historis ke dalam matriks beban pesan
    for ucapan_gpt, ucapan_lokal in zip(rekaman_interaksi_gpt, rekaman_interaksi_ollama):
        beban_pesan.append({"role": "assistant", "content": ucapan_gpt})
        beban_pesan.append({"role": "user", "content": ucapan_lokal})
    
    respons = openai.chat.completions.create(model="gpt-4.1-mini", messages=beban_pesan)
    return respons.choices[0].message.content

# 4. Mendefinisikan fungsi transmisi untuk Agen Ollama
def invoke_ollama_agent():
    beban_pesan = [{"role": "system", "content": profil_ollama}]
    for ucapan_gpt, ucapan_lokal in zip(rekaman_interaksi_gpt, rekaman_interaksi_ollama):
        beban_pesan.append({"role": "user", "content": ucapan_gpt})
        beban_pesan.append({"role": "assistant", "content": ucapan_lokal})
    
    # Menangkap interaksi definitif terbaru dari agen GPT agar dapat diberikan argumen sanggahan
    beban_pesan.append({"role": "user", "content": rekaman_interaksi_gpt[-1]}) 
    respons = ollama.chat.completions.create(model="llama3.2", messages=beban_pesan)
    return respons.choices[0].message.content

# 5. Eksekusi siklus percakapan simulasi secara terstruktur
for iterasi in range(3):
    print(f"\n=== Siklus Evaluasi Perdebatan Teknis ke-{iterasi + 1} ===")
    
    # Penugasan agen GPT untuk memberikan argumen berbasis pragmatisme bisnis
    giliran_gpt = invoke_gpt_agent()
    print(f"🤖 Agen GPT (Pendukung Kode Cepat - Sinis):\n{giliran_gpt}\n")
    rekaman_interaksi_gpt.append(giliran_gpt) # Menyimpan riwayat argumen ke dalam memori
    
    # Penugasan agen Ollama untuk merespons dengan argumen berbasis kualitas arsitektur
    giliran_ollama = invoke_ollama_agent()
    print(f"🦙 Agen Ollama (Pendukung Kode Rapih - Santun):\n{giliran_ollama}\n")
    rekaman_interaksi_ollama.append(giliran_ollama) # Menyimpan riwayat argumen ke dalam memori